# CavSU Chatbot — Naive Bayes Intent Classification

A lightweight, fast chatbot for **Cavite State University (CavSU)** using:
- **Naive Bayes** classifier for intent prediction (no API needed)
- **TF-IDF** vectorization for text encoding
- **Scikit-learn** for ML pipeline
- Intent patterns from `cavsu_intents.json`

## Advantages
✅ **Free** — No API costs  
✅ **Fast** — Instant predictions, no latency  
✅ **Offline** — Works without internet  
✅ **Lightweight** — ~1MB model  
✅ **Interpretable** — See which features matter  

In [ ]:
import subprocess
import sys

packages = ["scikit-learn", "nltk", "numpy", "pandas", "joblib"]

for package in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

print("✓ All dependencies installed")

In [ ]:
import json
import os
import random
import re
import pickle
from collections import defaultdict

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import joblib

import nltk
from nltk.stem import WordNetLemmatizer

# Download NLTK resources
for resource in ['punkt', 'wordnet', 'omw-1.4']:
    try:
        nltk.data.find(f'tokenizers/{resource}')
    except LookupError:
        nltk.download(resource, quiet=True)

lemmatizer = WordNetLemmatizer()

print(f"✓ NumPy {np.__version__}")
print(f"✓ Scikit-learn {pd.__version__}")
print(f"✓ NLTK {nltk.__version__}")

## 1. Load Intents from JSON

In [ ]:
# Load intents JSON
INTENTS_PATH = "data/cavsu_intents.json"

with open(INTENTS_PATH, "r", encoding="utf-8") as f:
    intents_data = json.load(f)

intents = intents_data["intents"]

# Extract patterns and labels
training_patterns = []
training_labels = []
responses_map = defaultdict(list)

for intent in intents:
    tag = intent["tag"]
    responses_map[tag] = intent["responses"]
    
    for pattern in intent["patterns"]:
        training_patterns.append(pattern)
        training_labels.append(tag)

print(f"✓ Loaded {len(intents)} intent categories")
print(f"✓ Total training patterns: {len(training_patterns)}")
print(f"\nIntent breakdown:")
for intent in intents:
    count = len(intent["patterns"])
    print(f"  [{intent['tag']:<20}] {count:>3} patterns")

## 2. Text Preprocessing

In [ ]:
def preprocess_text(text):
    """Lowercase, remove punctuation, tokenize, and lemmatize."""
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", "", text)  # Remove punctuation
    tokens = nltk.word_tokenize(text)
    lemmatized = [lemmatizer.lemmatize(token) for token in tokens]
    return " ".join(lemmatized)

# Preprocess all patterns
preprocessed_patterns = [preprocess_text(p) for p in training_patterns]

print(f"Sample preprocessing:")
for raw, processed, label in zip(training_patterns[:3], preprocessed_patterns[:3], training_labels[:3]):
    print(f"  Raw:       {raw}")
    print(f"  Processed: {processed}")
    print(f"  Label:     {label}\n")

## 3. Train Naive Bayes Classifier

In [ ]:
# Create sklearn pipeline: TF-IDF → Naive Bayes
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=500, lowercase=True, stop_words='english')),
    ('classifier', MultinomialNB(alpha=0.1))
])

# Train the model
print("Training Naive Bayes classifier...")
pipeline.fit(preprocessed_patterns, training_labels)
print("✓ Model trained")

# Evaluate on training data
y_pred = pipeline.predict(preprocessed_patterns)
accuracy = accuracy_score(training_labels, y_pred)
print(f"\nTraining Accuracy: {accuracy:.2%}")

print("\nClassification Report:")
print(classification_report(training_labels, y_pred))

## 4. Save Model Artifacts

In [ ]:
MODEL_DIR = "models"
os.makedirs(MODEL_DIR, exist_ok=True)

# Save model pipeline
joblib.dump(pipeline, os.path.join(MODEL_DIR, "cavsu_classifier.pkl"))

# Save responses map
with open(os.path.join(MODEL_DIR, "responses_map.json"), "w", encoding="utf-8") as f:
    json.dump(dict(responses_map), f, ensure_ascii=False, indent=2)

print(f"✓ Model saved to {MODEL_DIR}/")
print(f"  - cavsu_classifier.pkl ({os.path.getsize(os.path.join(MODEL_DIR, 'cavsu_classifier.pkl')) / 1024:.1f} KB)")
print(f"  - responses_map.json")

## 5. Chatbot Inference

In [ ]:
CONFIDENCE_THRESHOLD = 0.30  # Minimum confidence to accept prediction

class CavSUChatbot:
    """Naive Bayes-based intent classification chatbot."""
    
    def __init__(self, model_path, responses_path):
        self.pipeline = joblib.load(model_path)
        with open(responses_path, "r", encoding="utf-8") as f:
            self.responses_map = json.load(f)
        self.conversation_history = []
    
    def get_response(self, user_input, confidence_threshold=CONFIDENCE_THRESHOLD):
        """Predict intent and return a response."""
        # Preprocess
        clean_input = preprocess_text(user_input)
        
        # Predict
        predicted_intent = self.pipeline.predict([clean_input])[0]
        
        # Get confidence (probability of predicted class)
        proba = self.pipeline.predict_proba([clean_input])[0]
        predicted_idx = list(self.pipeline.classes_).index(predicted_intent)
        confidence = float(proba[predicted_idx])
        
        # Fallback if confidence too low
        if confidence < confidence_threshold:
            predicted_intent = "nlu_fallback"
            confidence = 0.0
        
        # Get random response from intent
        response = random.choice(self.responses_map[predicted_intent])
        
        return response, predicted_intent, confidence
    
    def chat(self, user_input):
        """Process user message and track conversation history."""
        response, intent, confidence = self.get_response(user_input)
        
        self.conversation_history.append({
            "user": user_input,
            "bot": response,
            "intent": intent,
            "confidence": confidence
        })
        
        return response, intent, confidence
    
    def get_history(self):
        return self.conversation_history
    
    def clear_history(self):
        self.conversation_history = []

# Initialize chatbot
chatbot = CavSUChatbot(
    os.path.join(MODEL_DIR, "cavsu_classifier.pkl"),
    os.path.join(MODEL_DIR, "responses_map.json")
)

print("✓ Chatbot initialized")

## 6. Test with Sample Queries

In [ ]:
test_questions = [
    "What are the admission requirements?",
    "Is there Computer Science at CavSU?",
    "How much is the tuition?",
    "When is the entrance exam?",
    "Where is CavSU located?",
    "Are there scholarships available?",
    "What facilities does CavSU have?",
    "Tell me about CavSU",
    "Hello!",
    "Thanks, bye!",
]

print(f"{'Question':<45} {'Intent':<20} {'Confidence':>10}")
print("-" * 80)

for q in test_questions:
    response, intent, conf = chatbot.chat(q)
    print(f"{q:<45} {intent:<20} {conf:>10.1%}")
    print(f"  → {response[:70]}...")
    print()

## 7. Interactive Chat

In [ ]:
print("\n" + "="*70)
print("  CavSU Virtual Assistant — Iskolar para sa Bayan!")
print("="*70)
print("Type your question (or 'quit'/'exit' to stop)\n")

while True:
    user_input = input("You: ").strip()
    
    if not user_input:
        continue
    
    if user_input.lower() in ("quit", "exit", "bye"):
        print("CavSU Bot: Goodbye! Good luck with your studies!")
        break
    
    response, intent, confidence = chatbot.chat(user_input)
    print(f"CavSU Bot [{intent} | {confidence:.0%}]: {response}\n")

## 8. Utilities & Analysis

In [ ]:
def show_conversation():
    """Display formatted conversation history."""
    history = chatbot.get_history()
    if not history:
        print("No conversation history yet.")
        return
    
    print("\n" + "="*70)
    print("CONVERSATION HISTORY")
    print("="*70 + "\n")
    
    for i, turn in enumerate(history, 1):
        print(f"[Turn {i}]")
        print(f"  👤 You:      {turn['user']}")
        print(f"  🤖 Bot:      {turn['bot']}")
        print(f"  Intent:     {turn['intent']} ({turn['confidence']:.1%} confidence)")
        print()

def export_history(filename="chat_history.json"):
    """Export conversation history to JSON."""
    history = chatbot.get_history()
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(history, f, ensure_ascii=False, indent=2)
    print(f"✓ Conversation exported to {filename}")

print("✓ Utilities loaded:")
print("  - show_conversation()          Show chat history")
print("  - export_history(filename)     Save chat as JSON")
print("  - chatbot.clear_history()      Clear chat history")

## Naive Bayes vs Other Approaches

| Aspect | Naive Bayes | Claude API | Neural Network |
|--------|-------------|-----------|----------------|
| **Cost** | FREE | $$ | FREE |
| **Speed** | Instant | 1-2s latency | Instant |
| **Internet** | ❌ Offline | ✅ Required | ❌ Offline |
| **Accuracy** | 95%+ | 99%+ | 98%+ |
| **Model Size** | 1 MB | API | 5-50 MB |
| **Setup** | Simple | 1 line | Complex |
| **Customization** | Easy | Limited | Easy |

### When to Use Each
- **Naive Bayes** ← *You are here* — Perfect for fixed intents, fast & free
- **Claude API** — When you need to generate unique, conversational responses
- **Neural Network** — When you have 1000+ patterns or need to learn complex patterns
